# Windowed & tiled reads

Read only the pixels you need instead of the whole raster — the basis of out-of-core and
tile-server workflows:

- **`read_array(window=[xoff, yoff, xsize, ysize])`** — a pixel window.
- **`read_part(bbox, bbox_crs=...)`** — a geographic bounding-box subset.
- **`get_tile(size)`** — iterate fixed-size tiles across the raster.
- **`get_block_arrangement()`** — the raster's native on-disk block layout.

## Setup

In [1]:
%matplotlib inline

import tempfile
from pathlib import Path

import numpy as np

DATA = Path('../../../examples/data')
if not DATA.exists():
    DATA = Path('examples/data')
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t3-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(str(DATA / 'geotiff' / 'sentinel_crop.tif'))
ds.shape, ds.epsg

2026-06-08 23:58:28 | INFO | pyramids.base.config | Logging is configured.


((1, 256, 256), 4326)

## A pixel window — `read_array(window=...)`

`[xoff, yoff, xsize, ysize]` in pixels — reads only that block.

In [3]:
patch = ds.read_array(window=[50, 50, 100, 100])
np.asarray(patch).shape

(100, 100)

## A geographic subset — `read_part`

Give a bounding box (with its CRS); the reader clips and reads just that area.

In [4]:
xmin, ymin, xmax, ymax = ds.bbox
dx, dy = (xmax - xmin) / 4, (ymax - ymin) / 4
part = ds.read_part((xmin + dx, ymin + dy, xmax - dx, ymax - dy), bbox_crs=ds.epsg)
np.asarray(part).shape

(129, 129)

## Iterate tiles — `get_tile`

Yields fixed-size windows, so you can stream a large raster tile by tile.

In [5]:
tiles = list(ds.get_tile(size=128))
len(tiles), tiles[0].shape

(4, (128, 128))

## Native block layout — `get_block_arrangement`

The blocks GDAL actually stores on disk — read along these for the least I/O.

In [6]:
blocks = ds.get_block_arrangement()
list(blocks.columns), len(blocks)

(['x_offset', 'y_offset', 'window_xsize', 'window_ysize'], 32)

## Notes

- For lazy, parallel block processing use `read_array(chunks=...)` — see the
  [Dask quickstart — Dataset](../dask/dataset.ipynb).
- See also: [Raster algebra](raster-algebra.ipynb) (`map_blocks` applies a function per tile).